In [ ]:
from langchain_docling.loader import DoclingLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import numpy as np

In [ ]:
FILE_PATH = r"docs\Transformers.pdf"

## Step 1: LOAD

In [ ]:
loader = DoclingLoader(FILE_PATH)

In [ ]:
# Load all documents
documents = loader.load()   # Loads all documents into the memory immediately and returns a list.

In [ ]:
# # For large datasets, lazily load documents
# # it is used when pdf is too large and cannot fit-in memory at once  
# for document in loader.lazy_load():   # Returns a generator that yields one document at a time.
#     print(document)

In [ ]:
for document in documents:
    print()
    print(document.page_content)

In [ ]:
# See which document contains which page of pdf
for i, document in enumerate(documents):
    print(f"\nDocument {i}")

    pages = []

    for item in document.metadata["dl_meta"]["doc_items"]:
        for prov in item["prov"]:
            pages.append(prov["page_no"])

    print("Pages:", sorted(set(pages)))

## Step 2: SPLITT

In [ ]:
# first convert the all documents into strinf as we have to pass it to text splitter (which takes atring)
full_text = "\n".join([doc.page_content for doc in documents])

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
texts = text_splitter.split_text(full_text)
print(texts)

## Step3: EMBEDDINGS

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)

In [ ]:
document_embeddings = embeddings.embed_documents(texts)
query = input("What are transformers?")
query_embedding = embeddings.embed_query(query)

In [ ]:
def cosine_similarities(vect1, vect2):
    dot_product = np.dot(vect1, vect2)
    norm1 = np.linalg.norm(vect1)
    norm2 = np.linalg.norm(vect2)
    return dot_product / (norm1 * norm2)


cosine_similarities(document_embeddings, document_embeddings)